# Environment

## Setup - imports

In [ ]:
#Setup
#import rasterio                    # For raster processing
from datetime import datetime      # For date and time operations
from scipy.signal import convolve2d
import h5py
from tqdm import tqdm
import math
import time

import json                        # For JSON data handling
import optuna                      # For hyperparameter optimization
import numpy as np                 # For numerical operations
import os                          # For operating system interactions

import pandas as pd                # For data manipulation and analysis
import geopandas as gpd            # For geospatial data operations

import mesa                        # For agent-based modeling
from mesa.datacollection import DataCollector  # For collecting model data
import random                      # For random number generation
import copy                        # For creating deep copies of objects
import seaborn as sns              # For statistical data visualization
import matplotlib.pyplot as plt    # For plotting
import matplotlib.animation as animation  # For creating animations
from matplotlib.patches import Ellipse  # For drawing ellipses
from matplotlib import patches     # For drawing shapes
#import pysal                       # For spatial analysis
#from sklearn.cluster import DBSCAN  # For density-based clustering
from scipy import linalg           # For linear algebra operations
from shapely.geometry import Point, Polygon  # For geometric operations
from scipy.optimize import minimize  # For optimization
from scipy.ndimage import distance_transform_edt  # For distance calculations





## Setup - env and extent

In [ ]:
# Read shapefile containing geographic boundaries
gdf = gpd.read_file(r"D:\itay\HN_ext.shp")

# Extract the coordinates from the geometry to define the study area bounds
bounds = gdf.total_bounds  # Returns [minx, miny, maxx, maxy]
extent = {
    'minx': bounds[0],
    'miny': bounds[1],
    'maxx': bounds[2],
    'maxy': bounds[3]
}
print(f"Extent: minx={bounds[0]}, miny={bounds[1]}, maxx={bounds[2]}, maxy={bounds[3]}")

# Define coordinate system (ITM - Israel Transverse Mercator)
itm_crs = "EPSG:2039"

# Set directory paths for outputs and temporary files
base_directory = r"D:\itay\Model_2024\base"
dump_directory = r"D:\itay\Model_2024\dump"

# Create directories if they don't exist
os.makedirs(base_directory, exist_ok=True)
os.makedirs(dump_directory, exist_ok=True)


# New cell size (larger for simulation efficiency)
new_cell_size = 250

# Calculate scale factor for resampling
scale_factor = new_cell_size / 12.5

# Calculate new dimensions after resampling
new_cols = 318
new_rows = 280

print(f"New dimensions: {new_rows} rows, {new_cols} columns")

## Import variables

In [ ]:
output_dir = r"D:\itay\ABM\Data"
# Create the filename with path
output_file = os.path.join(output_dir, "yearly_data_10_25.h5") #list of 75 lists in the following structure: [pasture_fit, agri_fit, rain_fit]
y_output = []
with h5py.File(output_file, 'r') as f:
    # Get number of groups from metadata
    num_groups = f['metadata'].attrs['num_groups']

    # Load each group
    for i in range(num_groups):
        group_arrays = []
        group = f[f'group_{i}']

        # Get arrays in this group
        num_arrays = len(group.keys())

        for j in range(num_arrays):
            # Load array
            array = np.array(group[f'array_{j}'])
            group_arrays.append(array)

        y_output.append(group_arrays)

In [ ]:
# Open the HDF5 file in read mode
permanent_results=[]
file_path = r"../Data/per_data_10_25.h5"
with h5py.File(file_path, 'r') as f:
    # Access metadata
    metadata = f['metadata']
    num_groups = metadata.attrs['num_groups']
    arrays_per_group = metadata.attrs['arrays_per_group']
    array_shape = metadata.attrs['array_shape']

    # Load the arrays from the groups

    for i in range(num_groups):
        group = f[f'group_{i+1}']
        group_arrays = [np.array(group[f'array_{j}']) for j in range(arrays_per_group)]
        permanent_results=group_arrays

# Yearly Function

In [ ]:
def Yi_params( i, all_outputs):
    """
    This function calculates yearly environmental and anthropogenic factors.
    It tracks resource pressure over time and generates stress zones for modeling.

    Parameters:
    - param10: Initial resource raster for the first year
    - i: Current year index
    - all_outputs: List of outputs from previous years

    Returns:
    - return_ras: Raster showing likelihood of returning to previously used locations
    - yi_hu_stressed_z: Raster showing human-stressed zones
    - past_years: NumPy array of accumulated past impacts
    """

    # Calculate inter-annual pressure on resources
    if i>0:
        # Initialize an empty array with the same shape as the last year's output
        past_years = np.zeros_like(all_outputs[-1])
        past_years = past_years.reshape(new_cols,new_rows)


        # Iterate over all previous years' outputs to calculate cumulative effects
        for idx, output in enumerate(all_outputs):
            # Calculate weights that diminish with time (older years have less influence)
            # Calculate weights that increase with recency
            # Use reversed index to make recent years have more influence
            reversed_idx = len(all_outputs) - 1 - idx  # This makes older years have larger idx values

            decay_factor = 0.7  # Adjust as needed: higher values = faster decay
            weight = (decay_factor ** reversed_idx) * 1  # Standard weight for low impact areas
            weight2 = (decay_factor ** reversed_idx) * 1.5  # Higher weight for high impact areas

            # Apply different weights based on impact intensity
            mask = output < 2           # Areas with low impact
            mask2 = output >= 2         # Areas with high impact

            # Accumulate weighted values for different impact zones
            past_years[mask] += weight * output[mask]
            past_years[mask2] += weight2 * output[mask2]



        radius_cells = 20  # Convert map units to cells 20 cells*250m cell_size=5km radius
        # Create a grid of coordinates
        y, x = np.ogrid[-radius_cells:radius_cells + 1, -radius_cells:radius_cells + 1]
        # Calculate distances from center
        dist = np.sqrt(x * x + y * y)
        # Create inverse distance weights (with power parameter)
        power = 2  # Adjust as needed: higher values = faster decay with distance
        weights = 1.0 / np.power(dist + 1, power)  # Adding 1 to avoid division by zero

        # Normalize weights to sum to 1
        weights_normalized = weights / np.sum(weights)

        # Apply the convolution
        last_year_output = convolve2d(past_years, weights_normalized, mode='same', boundary='fill', fillvalue=0)

    else:
        # For the first year, use the initial parameters provided
        last_year_output = np.zeros((318, 280))
        past_years=np.zeros((318, 280))
        zones=last_year_output

    mean_value = np.mean(last_year_output)

    # Handle different scenarios for stress zone calculation
    if float(mean_value) == 0:
        # If mean value is zero, assign a baseline value of 10 to all zones
        yi_hu_stressed_z = last_year_output + 10
    else:
        # Remove zero values from consideration
        z=np.where(last_year_output==0,np.nan,last_year_output)
        # Define fuzzy membership function (equivalent to FuzzyLinear(2,0))
        def fuzzy_linear(x, max_val, min_val):
            # Scale values between 0 and 1 based on their position in the range
            # Invert the scale since FuzzyLinear(2,0) means higher values have lower membership
            normalized = (max_val - x) / (max_val - min_val)
            # Clip to [0,1] range
            return np.clip(normalized, 0, 1)
        # Apply fuzzy logic to create a gradient of stress levels
        hs_FuzzyAlgorithm = fuzzy_linear(z,2,0)

        yi_hu_stressed_z = hs_FuzzyAlgorithm * 10
        # Fill null values with baseline value of 10
        yi_hu_stressed_z = np.where(np.isnan(yi_hu_stressed_z), 10, yi_hu_stressed_z)




    # Calculate likelihood to return to previously used locations
    return_arr = past_years * 5


    return return_arr, yi_hu_stressed_z, past_years

# Agent_init:

## Agent_funcs

In [ ]:
# Function to calculate the direction in which the agent will grow or expand.

# Function to set the agent's camp (stop moving and establish a position)
def set_camp(agent):
    # Get agent's current position (x, y)
    x, y = agent.pos

    # Decrease surplus and manpower once camp is set
    agent.surplus -= 1
    # agent.manpower -= 1 # Removed to prevent long-term population drain

    # Mark the cell in the target raster as the camp location
    agent.model.target_raster[to_numpy_y(agent.model, y), x] += 0.75
# Function to calculate the number of agents based on total area and carrying capacity
def Num_agents():
    total_area_sq_km = 3251.54  # The Negev Highlands (model 2024)

    # Calculate the maximum number of agents that the area can support
    Max_carry_agents = total_area_sq_km / 18  # 9 km² per member agent

    # Calculate the number of agents based on the carrying capacity
    carry_agents = Max_carry_agents / 20  # Max_carry_agents * 0.5

    # Return the number of agents as an integer
    return int(np.floor(carry_agents))

# Function to calculate the pasture suitability value based on environmental factors
def pasture_val(model, current_pos, r):
    # Get the neighborhood of the current position
    nbr = model.grid.get_neighborhood(current_pos, moore=True, include_center=False, radius=r)

    # Filter out any cells that are out of bounds or outside the region of interest
    nbr = [cell for cell in nbr if not model.grid.out_of_bounds(cell) and cell[0] < 280]

    # Avoid division by zero in the stress raster by replacing zeros with a small value
    stress_safe = np.where(model.stress_ras == 0, 0.01, model.stress_ras)

    # Calculate the inverse stress factor (higher stress means less suitable)
    inverse_stress = 1 / (stress_safe / 10)

    # Cap the stress value to a maximum of 9
    capped_stress = np.minimum((inverse_stress - 1), 9)

    # Calculate the final vegetation map by adjusting for stress
    veg_map = np.maximum((model.veg_ras * (1 - (capped_stress * 0.1))), 0)

    # Collect the environmental values of neighboring cells
    env_values = [
        veg_map[to_numpy_y(model, pos[1]), np.clip(pos[0], 0, 279)]  # Vegetation suitability
        for pos in nbr
    ]

    # Calculate the mean environmental value of the neighborhood
    tot_env_value = sum(env_values)

    # Return the mean environmental value
    return tot_env_value



## Agent class

In [ ]:
class Household_Agent(mesa.Agent):
    """
    Represents a household unit in the model.
    Each household manages resources, manpower, territory, and enclosures over time.

    Attributes:
    ----------
    flock_head : int
        Represents the livestock units available to the household.
    surplus : int
        Represents the resource surplus of the household (from trade, agriculture, exchange).
    manpower : int
        The number of humans in the household, representing its workforce.
    territory : list
        The territory currently occupied by the household.
    memory : list
        Record of past encampments.
    enc_memory : list
        Record of past enclosures.
    """

    def __init__(self, model, threshold, manpower=None, surplus=None, flock_head=None):
        super().__init__(model)

        # Core attributes of the household
        if flock_head is None:
            self.flock_head = 0
            for _ in range(10):
                # ~18 livestock units per person, ~6 people per family
                flock_n = np.floor(random.gauss(105, 15))
                self.flock_head += flock_n
        else:
            self.flock_head = flock_head

        if manpower is None:
            self.manpower = max(35, np.floor(self.flock_head / 20))
        else:
            self.manpower = manpower

        if surplus is None:
            self.surplus = 0
        else:
            self.surplus = surplus

        self.territory = None
        self.memory = []
        self.enc_memory = []
        self.threshold = threshold  # Threshold for resource-based decisions = 10
        self.own_suitability_raster = None

    def step(self):
        """Defines the actions the household takes in a simulation step."""
        # Update flock size based on environmental conditions
        self.update_flock_size()
        self.calc_surplus() # Calculate surplus from current resources
        self.manpower -= 10  # yearly movement cost
        herders = (self.flock_head // 75)
        self.manpower -= herders  # manpower for herding
        existing_enclosure = self.find_recent_enclosure() # Check for an existing enclosure within the territory

        # Environmental quality assessment
        current_env_value = env_mean_val(self.model, self.pos, 20)
        env_quality = current_env_value / 10.0  # Normalized to 0-1 range

        # Calculate prosperity index
        denominator = self.manpower + 10 + herders
        if denominator > 0:
            surplus_ratio = self.surplus / denominator
        else:
            surplus_ratio = 0
        #surplus_ratio = self.surplus / (self.manpower + 10 + herders)
        prosperity_index = surplus_ratio * env_quality
        is_prosperous = prosperity_index > 0.4
        if is_prosperous and self.model.year > 5:
            # Only proceed with enclosure-related actions if manpower and surplus meet thresholds
            if self.manpower >= 15 and self.surplus >= self.threshold:
                if existing_enclosure:
                    xn, yn = existing_enclosure
                    # Basic cost for reuse
                    manpower_cost = 10
                    surplus_cost = 6
                    # Reduce household resources
                    self.manpower = max(self.manpower - manpower_cost, 0)
                    self.surplus = max(self.surplus - surplus_cost, 0)
                    self.model.target_raster[to_numpy_y(self.model, yn), xn] += 1
                    self.model.enclosures.append([existing_enclosure, self.model.year, self])
                    self.enc_memory.append([existing_enclosure, self.model.year])
                    self.manpower += manpower_cost * 0.9
                elif prosperity_index > 0.5 and self.manpower >= 25 and self.surplus >= self.threshold * 2:
                    # Prosperous households can build new enclosures
                    enc_pos = self.build_enclosure(self.territory, current_env_value)
                    xn, yn = enc_pos
                    manpower_cost = 20
                    surplus_cost = 25
                    # More investment leads to better quality enclosures
                    enclosure_quality = 2 + (prosperity_index * 2)
                    self.manpower = max(self.manpower - manpower_cost, 0)
                    self.surplus = max(self.surplus - surplus_cost, 0)
                    self.model.target_raster[to_numpy_y(self.model, yn), xn] += enclosure_quality
                    self.model.enclosures.append([enc_pos, self.model.year, self])
                    self.enc_memory.append([enc_pos, self.model.year])
                    self.manpower += manpower_cost * 0.9

        # Handle survival crisis if surplus is too low
        if self.surplus < 10:
            survived = self.handle_survival_crisis(threshold=10)
            if not survived:
                self.remove()
                return
        # When households have excess surplus but small herds, reinvest in production
        if self.surplus > 80 and self.flock_head < 600:
            # Invest up to 20% of surplus in herd expansion
            investment_amount = min(self.surplus * 0.2, 40)
            purchased_livestock = self.buy_livestock_with_surplus(investment_amount)
        # Adjust manpower
        self.manpower = max(1, self.manpower + 10)
        self.manpower += herders
        if random.uniform(0, 1) > 0.8:
            self.manpower += math.ceil(5 * random.uniform(0, 1))
        if self.manpower > 35 and random.uniform(0, 1) > 0.8:
            self.manpower -= math.ceil(5 * random.uniform(0, 1))

        # Surplus-dependent manpower bonuses
        if self.surplus > 100:
            self.manpower += min(5, self.surplus // 20)

        # surplus decay
        # Surplus decay over time
        #if self.surplus > 50:
        #    base_decay_rate = 0.05  # 5% base decay
        #    # Increased decay rate for higher surplus
        #    scaling_factor = 0.003 * (self.surplus - 50)
        #    decay_rate = min(0.4, base_decay_rate + scaling_factor)  # Cap at 40%
        #    decay_amount = self.surplus * decay_rate
        #    self.surplus -= decay_amount
        #else:
        #    decay_rate=0.025
        #    decay_amount = self.surplus * decay_rate
        #    self.surplus -= decay_amount

    def year_initiation(self):
        """Initial setup for the household at the start of the year."""
        self.own_suitability_raster = None
        self.memory = [i for i in self.memory if (self.model.year - i[2]) * random.random() < 5]

        mesa_x, mesa_y = place_household(self.model, self, self.model.suitability_raster)
        self.model.grid.place_agent(self, (mesa_x, mesa_y))

        # Check environmental conditions
        current_env_value = env_mean_val(self.model, self.pos, 20)

        # Calculate territory radius based on environmental conditions
        base_radius = 20
        if current_env_value < 3.5:
            territory_radius = base_radius + 10
        elif current_env_value < 5.0:
            territory_radius = base_radius + 5
        else:
            territory_radius = base_radius

        self.territory = self.model.grid.get_neighborhood(
            (mesa_x, mesa_y), moore=True, include_center=True, radius=territory_radius)
        self.model.territories.append(self.territory)
        set_camp(self)

        for _ in range(9):
            selected_cell = place_members(self.model, self, self.territory)
            env_degrade(self.model, selected_cell[0], selected_cell[1], 11, 0.3, 0.5)
            self.memory.append([selected_cell, env_mean_val(self.model, selected_cell, 5), self.model.year])
            self.model.target_raster[to_numpy_y(self.model, selected_cell[1]), selected_cell[0]] += 0.5

        env_degrade(self.model, mesa_x, mesa_y, territory_radius, 0.5, 0.9)
        self.memory.append([self.pos, env_mean_val(self.model, self.pos, territory_radius), self.model.year])

    def calc_surplus(self):
        """
        surplus calculation.
        Surplus represents non-livestock economic resources (copper trade, agriculture, exchanges).

        """
        # Preserve current surplus
        current_surplus = self.surplus
        self.surplus = 0

        # ==============================================
        # AGRICULTURAL BENEFIT
        # ==============================================
        ag_values = []
        for cell in self.territory:
            x, y = cell
            ag_values.append(self.model.ag_ras[to_numpy_y(self.model, y), x])

        mean_ag_value = sum(ag_values) / len(ag_values) if ag_values else 0
        ag_benefit = min(0.4, mean_ag_value * 0.04)  # Up to 40% reduction

        # ==============================================
        # SUBSISTENCE REQUIREMENTS
        # ==============================================
        # Base subsistence need adjusted by agriculture
        base_need_per_person = 18 * (1 - ag_benefit)
        subsistence_need = self.manpower * base_need_per_person

        # ==============================================
        # CARRYING CAPACITY & STRESS
        # ==============================================
        env_value = pasture_val(self.model, self.pos, 25)
        neighborhood = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False, radius=25)
        neighborhood_size = len(neighborhood)
        avg_env_value = env_value / neighborhood_size if neighborhood_size > 0 else 0

        # Carrying capacity calculation
        quality_factor = avg_env_value / 5.0 # Medium quality (5 on 0-10 scale) with 250m*250m cells (0.0625 sq km)
        carrying_capacity = neighborhood_size * 1.125 * quality_factor # Each cell at medium quality supports 1.125 goats

        # Calculate stress ratio
        stress_ratio = max(0, 1 - (carrying_capacity / self.flock_head)) if self.flock_head > 0 else 0

        # ==============================================
        # TOTAL LIVESTOCK REQUIREMENT WITH BUFFER
        # ==============================================
        # Add 50% buffer so households aren't trimmed to bare minimum
        buffer_multiplier = 1.5
        stress_multiplier = 1 + (stress_ratio * 0.3)  # stress increases requirements

        total_livestock_needed = subsistence_need * buffer_multiplier * stress_multiplier
        if self.flock_head > 0:
            non_cull_fraction = max(0, (self.flock_head - total_livestock_needed) / self.flock_head)
        else:
            non_cull_fraction = 0

        # ==============================================
        # TWO TYPES OF LIVESTOCK-BASED SURPLUS GENERATION
        # ==============================================

        # TYPE 1: NON-LETHAL PRODUCTS (milk, wool, dung, etc.)
        # These come from the ENTIRE herd without killing animals
        # Productive animals generate these products annually

        productive_herd = max(0, self.flock_head - (subsistence_need*non_cull_fraction))  # animals beyond bare subsistence

        # Non-lethal products conversion rate (lower than culling)
        # Represents milk, wool, dung, occasional offspring sales
        non_lethal_rate = 0.15  # 1 productive animal → 0.15 surplus/year

        surplus_from_products = productive_herd * non_lethal_rate

        # TYPE 2: CULLING EXCESS ANIMALS
        # Only animals BEYOND the buffer are actually culled

        excess_livestock = max(0, self.flock_head - total_livestock_needed)

        # Culling conversion rate (higher than non-lethal products)
        # Represents meat, hides, etc. from slaughtered animals
        culling_rate = 0.35  # 1 culled animal → 0.35 surplus
        
        # Cull 80% of excess, keep 20% as additional buffer/wealth storage
        cull_amount = excess_livestock * 0.8
        surplus_from_culling = cull_amount * culling_rate

        # Remove the culled livestock
        self.flock_head -= cull_amount
        # TOTAL SURPLUS GENERATION
        total_surplus_generated = surplus_from_products + surplus_from_culling
        # DECAY
        decay_rate = 0.3  # 30% annual decay
        preserved_surplus = current_surplus * (1 - decay_rate)

        # Basic per-capita consumption
        consumption_per_person = 0.8
        
        # Calculate consumption based on need + small luxury factor (5%), rather than forced 80% depletion
        base_consumption = self.manpower * consumption_per_person
        luxury_consumption = (preserved_surplus + total_surplus_generated) * 0.05
        
        total_consumption = base_consumption + luxury_consumption

        # ==============================================
        # FINAL SURPLUS CALCULATION
        # ==============================================
        final_surplus = preserved_surplus + total_surplus_generated - total_consumption

        # Cap maximum (prevents unrealistic accumulation)
        max_surplus_cap = 200 + (self.manpower * 5)
        self.surplus = min(max(0, final_surplus), max_surplus_cap)

    def convert_livestock_to_surplus(self, amount_needed):
        """
        EMERGENCY CONVERSION: livestock → surplus
        Used when in crisis only.
        Represents selling livestock in trade networks.

        Args:
            amount_needed: Amount of surplus needed

        Returns:
            float: Amount of surplus actually gained
        """
        conversion_rate = 0.3  # 1 livestock → 0.2 surplus (selling under pressure)

        # Calculate livestock needed for conversion
        livestock_needed = math.ceil(amount_needed / conversion_rate)

        # Ensure household keeps minimum viable herd
        # Use the buffered subsistence calculation
        base_need_per_person = 18
        minimum_reserve = min(self.manpower * base_need_per_person, 100)
        available_livestock = max(0, self.flock_head - minimum_reserve)

        # Convert what's available
        livestock_to_convert = min(available_livestock, livestock_needed)

        if livestock_to_convert > 0:
            self.flock_head -= livestock_to_convert
            surplus_gained = livestock_to_convert * conversion_rate
            self.surplus += surplus_gained
            return surplus_gained

        return 0

    def buy_livestock_with_surplus(self, surplus_to_spend):
        """
        surplus → livestock
        Used to rebuild herds or expand production capacity.
        Represents purchasing livestock with trade income.

        Args:
            surplus_to_spend: Amount of surplus willing to spend

        Returns:
            int: Number of livestock purchased
        """
        conversion_rate = 0.3  # inverse: 0.3 surplus → 1 livestock pricier to acquire livestock than raise

        # Calculate how much livestock can be purchased
        livestock_per_surplus = 1 / conversion_rate  # 3 livestock per 1 surplus
        livestock_gained = math.floor(surplus_to_spend * livestock_per_surplus)

        # Calculate actual cost
        actual_cost = livestock_gained / livestock_per_surplus

        # Ensure household has surplus to spend
        if self.surplus >= actual_cost:
            self.surplus -= actual_cost
            self.flock_head += livestock_gained
            return livestock_gained

        return 0

    def update_flock_size(self):
        """
        Updates flock size based on environmental conditions and stress levels.
        """
        # Calculate environmental value
        env_value = pasture_val(self.model, self.pos, 30)

        # Get the neighborhood cells
        neighborhood = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False, radius=30)
        neighborhood_size = len(neighborhood)

        # Calculate average environmental value per cell
        avg_env_value = env_value / neighborhood_size if neighborhood_size > 0 else 0

        # Calculate carrying capacity
        quality_factor = avg_env_value / 5.0
        goats_per_cell = 1.125 * quality_factor
        carrying_capacity = goats_per_cell * neighborhood_size

        # Calculate resource-to-livestock ratio
        ratio = carrying_capacity / self.flock_head if self.flock_head > 0 else 2.0

        # Get average stress level in territory
        stress_values = []
        for cell in self.territory:
            x, y = cell
            numpy_y = to_numpy_y(self.model, y)
            if 0 <= numpy_y < self.model.stress_ras.shape[0] and 0 <= x < self.model.stress_ras.shape[1]:
                stress_values.append(self.model.stress_ras[numpy_y, x])

        avg_stress = sum(stress_values) / len(stress_values) if stress_values else 0
        stress_factor = 1 - min(0.5, avg_stress / 20)

        # Calculate small flock factor
        small_flock_factor = 1.0
        if self.flock_head < 500:
            small_flock_factor = 2.0 - (self.flock_head / 500)

        # Adjust flock size based on resource ratio and stress
        if ratio > 1.1:  # More resources than needed
            max_growth_rate = 0.2 * stress_factor * small_flock_factor # Increased base growth
            diminishing_factor = max(0.3, 1 - (self.flock_head / 2500)) # Relaxed cap
            growth_rate = max_growth_rate * diminishing_factor * min(ratio, 2.0) * random.uniform(0.7, 1.3)

            max_sustainable = carrying_capacity * 1.2
            growth_amount = min(
                self.flock_head * growth_rate,
                max_sustainable - self.flock_head
            )

            self.flock_head = np.floor(self.flock_head + max(0, growth_amount))

        elif ratio < 0.9:  # Not enough resources
            # Softened decline logic: proportional to the gap
            gap = 0.9 - ratio
            base_decline = 0.05 + (gap * 0.5) # Starts at 5%, scales with resource gap
            stress_impact = 1 + (avg_stress / 20)

            if self.flock_head < 100:
                base_decline *= max(0.4, self.flock_head / 100)

            decline_rate = base_decline * stress_impact * random.uniform(0.8, 1.2)
            decline_rate = min(decline_rate, 0.3) # Cap max decline to prevent total crash
            self.flock_head = np.floor(self.flock_head * (1 - decline_rate))

    def find_recent_enclosure(self):
        """Finds the most recent usable enclosure within the household's territory."""
        recent_enclosures = []
        current_year = self.model.year

        # Prioritize household's own enclosures
        for enc in self.enc_memory:
            if enc[0] in self.territory and (current_year - enc[1]) < 20:
                score = 20 - (current_year - enc[1])
                recent_enclosures.append((enc[0], score))

        # Check all enclosures in the model
        for enc in self.model.enclosures:
            if enc[0] in self.territory and (current_year - enc[1]) < 15 and enc[1] != current_year:
                score = (15 - (current_year - enc[1])) * 0.8
                recent_enclosures.append((enc[0], score))

        if not recent_enclosures:
            return None

        # Adjust scores with environmental factors
        enhanced_scores = []
        for pos, score in recent_enclosures:
            x, y = pos
            veg_value = self.model.veg_ras[to_numpy_y(self.model, y), x]
            ag_value = self.model.ag_ras[to_numpy_y(self.model, y), x]
            env_factor = 1.0 + ((veg_value * 0.02) + (ag_value * 0.02))
            final_score = score * env_factor * random.uniform(0.8, 1.2)
            enhanced_scores.append((pos, final_score))

        if not enhanced_scores:
            return None

        # Select probabilistically based on weighted scores
        selected_idx = random.choices(
            range(len(enhanced_scores)),
            weights=[s[1] for s in enhanced_scores],
            k=1
        )[0]
        return enhanced_scores[selected_idx][0]

    def handle_survival_crisis(self, threshold=10):
        """
        crisis response.

        Households convert livestock to surplus or vice versa when in crisis.
        Binary outcome: survive or fail.

        Args:
            threshold: Minimum surplus needed for survival

        Returns:
            bool: True if household survives, False if removed
        """
        if self.surplus >= threshold:
            if self.flock_head < 25:
                return True
            elif self.surplus>threshold*10:
                livestock_deficit = 30 - self.flock_head
                self.buy_livestock_with_surplus(livestock_deficit)
                return True

        # Calculate surplus deficit
        surplus_deficit = threshold - self.surplus
        self.convert_livestock_to_surplus(surplus_deficit)

        # Check survival conditions
        if self.surplus < 0 or self.flock_head < 15 or self.manpower <= 0:
            # Store resources for replacement household
            leftover_resources = {
                'manpower': max(1, self.manpower),
                'surplus': max(0, self.surplus),
                'flock_head': max(0, self.flock_head)
            }
            if not hasattr(self.model, 'scheduled_new_agents'):
                self.model.scheduled_new_agents = []
            self.model.scheduled_new_agents.append(leftover_resources)
            return False  # Household removed
        # Crisis households maintain normal territory size

        return True  # Household survives

    def build_enclosure(self, nbr, mean_val):
        """
        Selects the best cell for building an enclosure using probabilistic methods.
        """
        bst_cells = []

        # Recent own enclosures with higher weight
        for enc in self.enc_memory:
            if enc[0] in nbr:
                age = self.model.year - enc[1]
                xi, yi = enc[0]

                veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]

                if age < 15:
                    base_score = 20 - age
                else:
                    base_score = 5 + (20 - min(age, 20)) / 3

                env_factor = 1.0 + (veg_value * 0.02) + (ag_value * 0.02) + (suit_val * 0.02)
                total_score = base_score * env_factor

                ownership_bias = 1.5
                final_score = total_score * ownership_bias

                random_factor = random.uniform(0.9, 1.1)
                bst_cells.append([enc[0], final_score * random_factor])

        # Consider other enclosures
        for enc_data in self.model.enclosures:
            enc_pos, enc_year, enc_owner = enc_data
            if enc_pos in nbr and enc_pos not in [e[0] for e in self.enc_memory]:
                age = self.model.year - enc_year
                if age > 15:
                    xi, yi = enc_pos

                    veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                    ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                    suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]

                    base_score = 10 - (age / 2)
                    env_factor = 1.0 + (veg_value * 0.02) + (ag_value * 0.02) + (suit_val * 0.02)
                    final_score = base_score * env_factor

                    random_factor = random.uniform(0.7, 1.3)
                    bst_cells.append([enc_pos, final_score * random_factor])

        # Check reuse viability
        has_reuse_candidates = False
        if bst_cells:
            average_score = sum(cell[1] for cell in bst_cells) / len(bst_cells)
            if average_score > 8:
                has_reuse_candidates = True

        if self.model.enclosures:
            should_consider_new = not has_reuse_candidates or random.random() > 0.7
        else:
            should_consider_new = random.random() > 0.4

        if should_consider_new:
            # Consider new locations
            for cell in nbr:
                if any(e[0] == cell and (self.model.year - e[1]) < 25 for e in self.model.enclosures):
                    continue

                xi, yi = cell

                veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]

                env_score = (suit_val * 0.7) + (veg_value * 0.15) + (ag_value * 0.15)

                if env_score > mean_val + 1:
                    base_score = env_score * 0.8
                    random_factor = random.uniform(0.75, 1.25)
                    bst_cells.append([cell, base_score * random_factor])

        if not bst_cells:
            return self.pos
        else:
            weights = [max(0, cell[1]) for cell in bst_cells]
            total_weight = sum(weights)

            if total_weight > 0:
                normalized_weights = [w / total_weight for w in weights]
                selected_idx = random.choices(range(len(bst_cells)), weights=normalized_weights, k=1)[0]
                selected_cell = bst_cells[selected_idx][0]
            else:
                if bst_cells:
                    selected_idx = random.randint(0, len(bst_cells) - 1)
                    selected_cell = bst_cells[selected_idx][0]

        return selected_cell

    def update_own_suitability_raster(self):
        """Updates household's personalized suitability raster based on territory and memory."""
        if self.own_suitability_raster is None:
            self.own_suitability_raster = self.model.suitability_raster.copy()

        # Adjust based on territory
        if self.territory:
            territory_mask = np.zeros_like(self.own_suitability_raster, dtype=bool)
            for x, y in self.territory:
                numpy_y = to_numpy_y(self.model, y)
                territory_mask[numpy_y, x] = True

            distances = distance_transform_edt(~territory_mask)
            adjustment = np.where(distances <= 10, 1 / (distances + 1), 0)
            self.own_suitability_raster += adjustment

        # Adjust based on memory
        if self.memory:
            for (x, y), value, year in self.memory:
                numpy_cell = (to_numpy_y(self.model, y), x)
                diff_y = self.model.year - year
                if diff_y == 0:
                    self.own_suitability_raster[numpy_cell] -= 0.5
                else:
                    time_factor = 1 / diff_y
                    if value >= 5:
                        self.own_suitability_raster[numpy_cell] += value * time_factor * 0.2
                    elif value > 0:
                        self.own_suitability_raster[numpy_cell] -= (1 / value) * time_factor * 0.2

# Model_init

## Model_funcs

In [ ]:
# in-model functions
# Convert Mesa y-coordinate to NumPy y-coordinate (Mesa's y-axis is inverted)
def to_numpy_y(model, mesa_y):
    numpy_y = np.clip(model.grid.height - mesa_y - 1, 0, model.grid.height - 1)
    return numpy_y

# Calculate the mean environmental suitability value in a neighborhood
def env_mean_val(model, current_pos, r):
    # Get neighborhood cells within radius r
    nbr = model.grid.get_neighborhood(
        current_pos, moore=True, include_center=False, radius=r
    )
    # Filter out-of-bounds cells and cells beyond x=280
    nbr = [cell for cell in nbr if not model.grid.out_of_bounds(cell) and cell[0] < 280]
    # Extract suitability values for each cell in the neighborhood
    env_values = [
        model.suitability_raster[to_numpy_y(model, pos[1]), np.clip(pos[0], 0, 279)]
        for pos in nbr
    ]
    # Calculate the mean suitability value
    mean_env_value = sum(env_values) / len(env_values)
    return mean_env_value

# Calculate the carrying capacity of the environment based on suitability and population needs
def calculate_carrying_capacity(model, suitability_raster, population, needs):
    """Calculates the carrying capacity based on average suitability."""
    max_agent_resources = population * needs  # Total resources needed by the population
    carrying_capacity = np.sum(suitability_raster) / max_agent_resources  # Total suitability divided by needs
    return int(np.floor(carrying_capacity))  # Round down to the nearest integer

# Evaluate the suitability of a neighborhood for placing a household
def eval_neighborhood(model, x, y, household, radius, n):
    # Get neighborhood cells within the given radius
    nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=radius)
    # Count non-empty cells in the neighborhood
    for cell in nbr:
        if model.grid.is_cell_empty(cell) is False:
            n += 1
    # Convert y-coordinate to NumPy format
    numpy_y = model.suitability_raster.shape[0] - y - 1
    # Define boundaries for the subarray (neighborhood area)
    start_x = max(0, x - radius)
    stop_x = min(model.suitability_raster.shape[1], x + radius + 1)
    start_numpy_y = max(0, numpy_y - radius)
    stop_numpy_y = min(model.suitability_raster.shape[0], numpy_y + radius + 1)
    # Extract subarrays for suitability and return rasters
    subarray = model.suitability_raster[start_numpy_y:stop_numpy_y, start_x:stop_x]
    ret_array = model.return_raster[start_numpy_y:stop_numpy_y, start_x:stop_x]
    # Adjust suitability values if return raster values are >= 2
    if np.any(ret_array >= 2):
        mask2 = ret_array >= 2
        subarray[mask2] += ret_array[mask2]
    # Calculate the carrying capacity for the neighborhood
    eval_env = calculate_carrying_capacity(model, subarray, household.manpower, n)
    return eval_env

# Check if a neighborhood overlaps with another household's territory
def overlap_territory(model, current_agent, nbr):
    l = []
    Overlap = 0
    # Iterate through all households except the current one
    for household in model.agents_by_type[Household_Agent]:
        if household != current_agent and household.territory != None:
            # Check if any cell in the neighborhood overlaps with the household's territory
            if any(cell in household.territory for cell in nbr):
                Overlap += 1
            else:
                Overlap += 0
            # Calculate the overlap ratio
            overlap_state = Overlap / (len(nbr))
            # Return True if overlap exceeds 25%
            if overlap_state > 0.25:
                return True
            else:
                return False

# Place a household agent in the environment based on suitability and memory
def place_household(model, current_agent, suitability_raster):
    # Create a copy of the suitability raster to avoid modifying the original
    current_agent.update_own_suitability_raster()
    # Generate probability weights for placement
    current_agent.own_suitability_raster[current_agent.own_suitability_raster < 0]=0
    prob_ras=model.place_raster * current_agent.own_suitability_raster
    prob_weights = current_agent.own_suitability_raster.flatten() ** 3  # Weight by suitability cubed
    prob_weights = np.nan_to_num(prob_weights, nan=0.0, posinf=0.0, neginf=0.0)

    # Normalize probabilities
    total_weight = np.sum(prob_weights)
    if total_weight > 0:
        prob = prob_weights / total_weight
    else:
        # Fallback to uniform distribution
        prob = np.ones_like(prob_weights) / len(prob_weights)
    # Ensure sum is exactly 1
    prob /= prob.sum()

    # Function to find a valid position based on probabilities
    def get_valid_position():
        while True:
            # Randomly select a position based on probabilities
            random_index = np.random.choice(np.arange(len(prob)), p=prob)
            y, x = np.unravel_index(random_index, suitability_raster.shape)
            mesa_y = suitability_raster.shape[0] - y - 1
            # Check if the position and its neighborhood are within bounds
            if (20 <= x < model.grid.width - 20 and
                20 <= mesa_y < model.grid.height - 20 and
                20 <= y < suitability_raster.shape[0] - 20 and
                20 <= x < suitability_raster.shape[1] - 20 and
                model.place_raster[y, x] == 1):
                return x, mesa_y

    # Get initial position
    x, y = get_valid_position()
    nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=20)

    # Keep trying new positions until one satisfies all conditions
    while (overlap_territory(model, current_agent, nbr) and
           eval_neighborhood(model, x, y, current_agent, 20, 35) < 1):
        x, y = get_valid_position()
        nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=20)

    return (x, y)

# Place household members within a specified territory
def place_members(model, current_agent,  ter):
    # Create a copy of the suitability raster and mask forbidden areas
    current_agent.update_own_suitability_raster()
    own_suitability_raster=current_agent.own_suitability_raster
    # Filter territory cells to include only valid positions
    ter = [cell for cell in ter if model.place_raster[to_numpy_y(model, cell[1]), cell[0]]]
    # Extract suitability values for the territory
    suitability_values = [
        own_suitability_raster[to_numpy_y(model, cell[1]), cell[0]] for cell in ter
    ]

    # Generate probability weights (cubed to emphasize high suitability)
    prob_weights = np.array(suitability_values) ** 3
    prob_weights = np.nan_to_num(prob_weights, nan=0.0, posinf=0.0, neginf=0.0)
    total_weight = np.sum(prob_weights)

    # Normalize probabilities
    if total_weight > 0:
        prob = prob_weights / total_weight
    else:
        prob = np.ones(len(ter)) / len(ter)
    # Ensure sum is exactly 1
    prob /= prob.sum()
    # Select a cell within the territory based on probabilities
    selected_index = np.random.choice(len(ter), p=prob)
    selected_cell = ter[selected_index]
    return selected_cell  # Return the selected cell coordinates

# Simulate environmental degradation around a position
def env_degrade(model, mesa_x, mesa_y, r, d_factor, p_factor):
    """
    Optimized version of environmental degradation calculation.
    Uses vectorized operations where possible and pre-filters invalid cells.

    Args:
        model: Model instance containing the grid and suitability raster
        mesa_x, mesa_y: Position coordinates in Mesa system
        r: Radius for neighborhood calculation
        d_factor: Distance-based degradation factor
        p_factor: Position-based degradation factor
    """
    # Convert Mesa y-coordinate to NumPy y-coordinate
    numpy_y = model.grid.height - mesa_y - 1

    # Get neighborhood cells within the given radius
    nbr = model.grid.get_neighborhood((mesa_x, mesa_y), moore=True, include_center=False, radius=r)

    # Convert neighborhood cells to NumPy arrays for vectorized operations
    cells = np.array(nbr)
    x_coords = cells[:, 0]
    y_coords = model.grid.height - cells[:, 1] - 1  # Transform all y-coordinates at once

    # Create a mask for valid coordinates
    valid_mask = (
        (x_coords >= 0) &
        (x_coords < model.suitability_raster.shape[1]) &
        (y_coords >= 0) &
        (y_coords < model.suitability_raster.shape[0])
    )

    # Filter valid coordinates
    valid_x = x_coords[valid_mask]
    valid_y = y_coords[valid_mask]
    original_y = cells[valid_mask][:, 1]  # Original Mesa y-coordinates for distance calculation

    # Calculate distances vectorized
    x_distances = np.abs(mesa_x - valid_x)
    y_distances = np.abs(mesa_y - original_y)
    distance_factors = d_factor / ((x_distances + y_distances) / 2)

    # Update suitability values vectorized
    current_values = model.suitability_raster[valid_y, valid_x]
    new_values = np.maximum(current_values - distance_factors, 0.0001)  # Ensure values don't drop below 0.0001
    model.suitability_raster[valid_y, valid_x] = new_values

    # Update the center position with a higher degradation factor
    model.suitability_raster[numpy_y, mesa_x] = max(
        model.suitability_raster[numpy_y, mesa_x] - p_factor,
        0.0001
    )

In [ ]:
def viz_map(model, suitability_raster, year, run_dir):
    """
    Visualize the model's state, including suitability raster and agent positions.
    Saves the figure without displaying it during runtime.

    Args:
        model: The simulation model instance.
        suitability_raster: The environmental suitability raster.
        year: Current simulation year.
        run_dir: Directory to save visualization.
    """
    # Set up visualization theme and figure
    sns.set_theme(style="whitegrid")
    height, width = suitability_raster.shape

    # Create figure with the plt.ioff() to prevent display
    plt.ioff()  # Turn off interactive mode
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the suitability raster with fixed scale 0-10
    im = ax.imshow(suitability_raster, cmap="YlGn", origin="upper", vmin=0, vmax=10)

    # Add colorbar for the scale
    cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Suitability Index (0-10)')

    # Assign unique colors to households
    household_dict = {}
    color_palette = sns.color_palette("tab20", n_colors=len(model.agents_by_type[Household_Agent]))
    color_index = 0

    # Plot household agents
    for agent in model.agents_by_type[Household_Agent]:
        numpy_row = height - 1 - agent.pos[1]
        numpy_column = agent.pos[0]
        agent_color = color_palette[color_index]
        sns.scatterplot(x=[numpy_column], y=[numpy_row], color=agent_color, s=20, marker="o", ax=ax)
        household_dict[agent] = agent_color
        color_index += 1

    # Plot household members
    for agent in model.agents_by_type[Household_Agent]:
        for c in agent.memory:
            if c[-1] == year:
                numpy_row = height - 1 - c[0][1]
                numpy_column = c[0][0]
                a_color = household_dict[agent]
                sns.scatterplot(x=[numpy_column], y=[numpy_row], color=a_color, s=10, marker="o", ax=ax)

    # Plot enclosures
    for enc in model.enclosures:
        numpy_row = height - 1 - enc[0][1]
        numpy_column = enc[0][0]
        #Check if the agent exists in household_dict
        if enc[2] in household_dict:
            enc_color = household_dict[enc[2]]
        else:
            # Use a default color if agent not found
            enc_color = 'gray'
        if year > 0:
            opc = (1 - (year - enc[1]) / year)
        else:
            opc = 1
        sns.scatterplot(x=[numpy_column], y=[numpy_row], color=enc_color, s=15, marker="s", ax=ax, alpha=opc)

    # Format title and axes
    plt.title(f"Year {year} Simulated Map", fontsize=14)
    plt.xticks([])
    plt.yticks([])

    # Add grid
    ax.grid(True, linestyle='--', alpha=0.7)

    # Save the visualization
    filename = f'year_{year}_map.png'
    filepath = os.path.join(run_dir, filename)
    plt.savefig(filepath, dpi=400)

    # Close the figure to free memory
    plt.close(fig)

## init

### weights

In [ ]:
# Find the most recent run folder
run_folder = max([f.path for f in os.scandir(r'../Results/opt/trial_0') if f.is_dir()])

# Load permanent weights from the previous run
with open(os.path.join(run_folder, 'permanent_weights_dict.json')) as f:
    permanent_weights_dict = json.load(f)

# Load yearly weights from the previous run
with open(os.path.join(run_folder, 'yearly_weights_dict.json')) as f:
    yearly_weights_dict = json.load(f)

In [ ]:
# Combine permanent and yearly weights into a single dictionary
weights = {**permanent_weights_dict, **yearly_weights_dict}
wts = list(weights.values())  # Extract weight values
sw = sum(wts)

# Normalize the weights
weights = {key: value / sw for key, value in weights.items()}  # Normalize each weight to sum to 1
wts=list(weights.values())
# Function to plot weights as a bar chart
def plot_weights(weights):
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(weights.keys())  # Factor names
    values = list(weights.values())  # Weight values

    # Create bar chart
    bars = ax.bar(names, values)
    ax.set_ylabel('Normalized Weight')
    ax.set_title('Normalized Weights for Different Factors')
    plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability

    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height,
                f'{height:.3f}',
                ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

# Plot the normalized weights
plot_weights(weights)

{'dist_to_kb': 1, 'p_water': 1, 'Mean_rain': 2, 'slope_suitability': 4, 'return_to_site': 3, 'humen_stress': 2, 'Yearly_rain': 4

In [ ]:
# Create dictionaries to store weights for permanent and yearly factors
permanent_weights_dict = {
    #"p_agri": wp1,  # Agricultural suitability
    "dist_to_kb": 1,  # Distance to Kadesh Barnea
    #"aspect": wp3,  # Terrain aspect
    "p_water": 1,  # Permanent water sources
    #"p_veg_fit": wp5,  # Vegetation suitability
    "Mean_rain": 1,  # Mean annual rainfall
    #"slope_range": wp7,  # Slope range
    "slope_suitability": 3,  # Slope suitability
}

yearly_weights_dict = {
    "return_to_site": 5,  # Return to previous sites
    "humen_stress": 4,  # Human stress
    #"pasture_y": ws3,  # Pasture yield
    #"yearly_agri": ws4,  # Yearly agricultural suitability
    #"yearly_water": ws5,  # Yearly water availability
    "Yearly_rain": 6,  # Yearly rainfall
}

In [ ]:
wts

### suitability

In [ ]:
in_dir= r"D:\itay\ABM\Data"
ext_file = os.path.join(in_dir, "ext_raster.npy")
place_file = os.path.join(in_dir, "place_raster.npy")
ext_raster=np.load(ext_file)
place_raster=np.load(place_file)
place_raster=place_raster[0:318,0:280]
ext_raster=ext_raster[0:318,0:280]


In [ ]:
def get_suitability_raster(y_output, indices, year, yearly_outputs, weights):
    """
    Generates a suitability raster for a given year based on factors and weights.

    Args:
        y_output: Output data for yearly factors.
        indices: Indices to access specific years in `y_output`.
        year: The current year of the simulation.
        yearly_outputs: All output data for the simulation.
        weights: List of weights for environmental factors.

    Returns:
        suitability_raster: A NumPy array representing the suitability raster.
        past_years: Data related to past years.
        yi_hu_stressed_z: Human stress data for the current year.
    """
    # Unpack the rasters into named variables for easier reference
    agri_raster, kb_suitability_raster, pw_suitability_raster, veg_fit, rain_suitability_raster,  slp_suitability_raster =permanent_results
    # Select the raster data for the current year
    random_year = y_output[indices[year]]

    # Unpack weights for permanent and yearly factors
    wp2,wp4, wp6,wp8, ws1, ws2, ws6 = weights

    # Unpack yearly factors from the selected year's data
    calc_pastoral_Yi, agr_raster,  YrainFuzzyMember_calc = random_year

    # Calculate additional yearly parameters (e.g., return raster, human stress)
    return_ras, yi_hu_stressed_z, past_years = Yi_params( year, yearly_outputs)
    # Sanitize inputs to prevent RuntimeWarnings (NaN/Inf)
    kb_suitability_raster = np.nan_to_num(kb_suitability_raster, posinf=0, neginf=0)
    pw_suitability_raster = np.nan_to_num(pw_suitability_raster, posinf=0, neginf=0)
    rain_suitability_raster = np.nan_to_num(rain_suitability_raster, posinf=0, neginf=0)
    slp_suitability_raster = np.nan_to_num(slp_suitability_raster, posinf=0, neginf=0)
    return_ras = np.nan_to_num(return_ras, posinf=0, neginf=0)
    yi_hu_stressed_z = np.nan_to_num(yi_hu_stressed_z, posinf=0, neginf=0)
    YrainFuzzyMember_calc = np.nan_to_num(YrainFuzzyMember_calc, posinf=0, neginf=0)
    # Summarize yearly factors using their weights
    res = ((wp2*kb_suitability_raster)+(wp4*pw_suitability_raster)+(wp6*rain_suitability_raster)+(wp8*slp_suitability_raster)+(ws1*return_ras)+(ws2*yi_hu_stressed_z)+(ws6*YrainFuzzyMember_calc))
    slp_lim = np.where(slp_suitability_raster<1, 0, 1)
    suitability_raster=res*slp_lim*ext_raster

    return suitability_raster, past_years, yi_hu_stressed_z



### Model class

In [ ]:
class NomadModel(mesa.Model):
    """
    The main model class of the simulation.
    Manages agents, the environment, and the simulation steps.
    """
    def __init__(self, suitability_raster, return_raster, stress_ras, inds, place_raster=place_raster, ras_w=wts):
        """
        Initialize the model with rasters, weights, and agents.

        Args:
            suitability_raster: Raster representing environmental suitability.
            return_raster: Raster representing return-to-site suitability.
            stress_ras: Raster representing human stress.
            inds: Indices for accessing yearly data.
            place_raster: Raster defining valid placement areas.
            ras_w: List of weights for environmental factors.
        """
        super().__init__()
        self.num_agents = Num_agents()  # Number of agents in the model
        height, width = suitability_raster.shape  # Dimensions of the raster
        self.grid = mesa.space.MultiGrid(width, height, False)  # MultiGrid for agent placement
        self.yearly_outputs = []  # Store yearly outputs for analysis
        self.suitability_raster = suitability_raster  # Environmental suitability raster
        self.target_raster = np.zeros_like(self.suitability_raster)  # Target raster for calculations
        self.return_raster = return_raster  # Return-to-site suitability raster
        self.place_raster = place_raster  # Raster defining valid placement areas
        self.ras_control = []  # Store copies of suitability raster for debugging
        self.weights = ras_w  # Weights for environmental factors
        self.year = 0  # Current simulation year
        self.territories = []  # List of territories claimed by households
        self.enclosures = []  # List of enclosures built by households
        self.suitability_raster[np.isnan(self.suitability_raster)] = 0  # Replace NaN values with 0
        self.threshold = 10  # Surplus Threshold for decision-making
        self.scheduled_new_agents = []  # Initialize the list for scheduled agent creations
        self.veg_ras = y_output[inds[0]][0]
        self.ag_ras = y_output[inds[0]][1]
        self.stress_ras = stress_ras
        self.ras_control.append(self.suitability_raster.copy())  # Store initial suitability raster

        # Create and place agents
        for i in range(self.num_agents):

            # Create the household agent
            agent = Household_Agent(self, threshold=self.threshold)
            mesa_x, mesa_y = place_household(self, agent, self.suitability_raster)  # Place household
            self.grid.place_agent(agent, (mesa_x, mesa_y))
            self.target_raster[to_numpy_y(self, agent.pos[1]), agent.pos[0]] += 0.75
            territory = self.grid.get_neighborhood((mesa_x, mesa_y), moore=True, include_center=True, radius=25)
            agent.territory = territory  # Assign territory to the household
            self.territories.append(agent.territory)

            for _ in range (9):
                selected_cell = place_members(self, agent,agent.territory)  # Place member
                env_degrade(self, selected_cell[0], selected_cell[1], 11, 0.3, 0.5)  # Simulate member degradation
                agent.memory.append([selected_cell, env_mean_val(self, selected_cell, 5), self.year])
                self.target_raster[to_numpy_y(self, selected_cell[1]), selected_cell[0]] += 0.5
            env_degrade(self, mesa_x, mesa_y, 25, 0.5, 0.9)  # Simulate household level environmental degradation
            agent.memory.append([agent.pos, env_mean_val(self, agent.pos, 20), self.year])  # Store memory



        # Set up DataCollector for agent data
        self.datacollector = DataCollector(agent_reporters={"Manpower": "manpower","flocks":"flock_head",
                                            "surplus": "surplus","position": "pos","enclosures": lambda a: list(a.enc_memory)})
        self.datacollector.collect(self)  # Collect initial data
        #print('done_init')

    def step(self):
        """
        Advance the model by one step (year).
        """
        self.agents_by_type[Household_Agent].shuffle_do("step")  # Step for household agents
        self.yearly_outputs.append(self.target_raster.copy())  # Store yearly output
        self.year += 1  # Increment year
        self.datacollector.collect(self)  # Collect data for the current step
        #print('done_step')

    def move_year(self, inds):
        """
        Move the model to the next year by updating rasters and resetting positions.
        """
        self.reset_pos()  # Reset agent positions
        # Update suitability, return, and stress rasters for the new year
        self.suitability_raster, self.return_raster, stress_ras = get_suitability_raster(y_output, inds, self.year, self.yearly_outputs, self.weights)
        self.ras_control.append(self.suitability_raster.copy())  # Store updated suitability raster
        self.veg_ras = y_output[inds[self.year]][0]
        self.ag_ras = y_output[inds[self.year]][1]
        self.stress_ras = stress_ras
        self.target_raster = np.zeros_like(self.suitability_raster)  # Reset target raster
        if hasattr(self, 'scheduled_new_agents') and self.scheduled_new_agents:
            for resources in self.scheduled_new_agents:
                self.create_replacement_agent(resources)
            self.scheduled_new_agents = []  # Reset the scheduled agent creations
        self.agents_by_type[Household_Agent].shuffle_do("year_initiation")  # Initialize new year for households

    def reset_pos(self):
        """
        Reset positions of all agents.
        """
        for agent in self.agents:
            self.grid.remove_agent(agent)  # Remove agents from the grid
    def create_replacement_agent(self, resources):
        """
        Creates a new agent with resources from a removed agent plus additional flocks.
        Uses Mesa 3.0's create_agents method.

        Parameters:
        ----------
        resources : dict
            Dictionary containing 'manpower', 'surplus', and 'flock_head' values from the removed agent
        """
        # Calculate additional flocks to add
        additional_flocks = sum(np.floor(random.gauss(90, 15)) for _ in range(10))

        # Create a new agent using Mesa's create_agents method
        new_agents = Household_Agent.create_agents(
            model=self,
            n=1,
            threshold=self.threshold,
            manpower=resources['manpower'],
            surplus=resources['surplus'],
            flock_head=resources['flock_head'] + additional_flocks
        )

        # Get the newly created agent
        agent = new_agents[0]
        #print(f"Created new agent {agent.unique_id} with manpower={agent.manpower}, flocks={agent.flock_head}, surplus={agent.surplus}")

# Model_exec

In [ ]:
def run_model(model_years, wts, y_output=y_output):
    """
    Runs the simulation for a specified number of years.

    Args:
        model_years: Number of years to simulate.
        wts: List of weights for environmental factors.
        y_output: Yearly output data for environmental factors.

    Returns:
        model: The completed model instance.
    """
    # Turn off interactive plotting globally at the beginning
    plt.ioff()

    # Generate a timestamp for the run directory
    current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    # Create a list of indices and shuffle them for random yearly data selection
    inds = list(range(len(y_output)))
    random.shuffle(inds)

    # Generate suitability, return, and stress rasters for the initial year
    suitability_raster, return_raster, stress_ras = get_suitability_raster(y_output, inds, year=0, yearly_outputs=[], weights=wts)

    # Initialize the model with the generated rasters
    model = NomadModel(suitability_raster, return_raster, stress_ras, inds)

    # Create a directory for storing run results
    run_directory = r"D:\itay\ABM\Results\runs_simp\run_{}".format(current_date)
    if not os.path.exists(run_directory):
        os.makedirs(run_directory)

    # Run the simulation for the specified number of years with progress bar
    for i in tqdm(range(model_years), desc="Simulating years"):
        model.step()  # Advance the model by one year
        if i != (model_years - 1):
            viz_map(model, model.suitability_raster, i, run_directory)
            model.move_year(inds)
        else:
            viz_map(model, model.suitability_raster, i, run_directory)

    # Retrieve collected data from the model
    household_data = model.datacollector.get_agent_vars_dataframe()

    # Save data to CSV files
    household_data.to_csv(os.path.join(run_directory, "household_data.csv"))

    # Save weights to JSON files
    with open(os.path.join(run_directory, "permanent_weights_dict.json"), "w") as f:
        json.dump(permanent_weights_dict, f)
    with open(os.path.join(run_directory, "yearly_weights_dict.json"), "w") as f:
        json.dump(yearly_weights_dict, f)

    # Turn interactive mode back on at the end if needed for other visualizations
    plt.ion()

    return model

In [ ]:
model=run_model(75,wts)

In [ ]:
model_gdf=to_gdf(model)
rd=r"D:\itay\ABM\Results\runs_simp\run_2026-01-07_17-48-12"
obj_func(gdf_real, model_gdf, rd)

## viz

In [ ]:
# For your agent data
print(household_data.describe())

In [ ]:

# For the first few rows to see structure
print(household_data.head())

In [ ]:
# Get the data for all household members
def plot_all_surplus(data):
    # Get the data for all household members

    # Create the plot
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")

    # Plot individual lines for each agent's surplus
    sns.lineplot(data=data, x="Step", y="surplus", alpha=0.3, color='gray', units="AgentID", estimator=None)

    # Add a line for the mean surplus
    sns.lineplot(data=data, x="Step", y="surplus", color='red', label='Mean Surplus', linewidth=2)

    plt.title("Changes in Household Members' Surplus Over Time")
    plt.xlabel("Step")
    plt.ylabel("Surplus")
    plt.show()
household_data=model.datacollector.get_agent_vars_dataframe()
plot_all_surplus(household_data)

In [ ]:
# Reset index to make Step and AgentID regular columns if needed
household_data_reset = household_data.reset_index()

# Create the plot
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Plot individual lines for each agent's flocks
sns.lineplot(data=household_data_reset, x="Step", y="flocks", alpha=0.3, color='gray',
             units="AgentID", estimator=None, legend=False)

# Calculate mean and std for each step
stats = household_data_reset.groupby("Step")["flocks"].agg(['mean', 'std']).reset_index()

# Plot mean line
plt.plot(stats["Step"], stats["mean"], color='blue', label='Mean Flocks', linewidth=2)

# Add standard deviation bands
plt.fill_between(stats["Step"],
                 stats["mean"] - stats["std"],
                 stats["mean"] + stats["std"],
                 alpha=0.3, color='lightblue', label='±1 Std Dev')

plt.title("Changes in Household Flocks Over Time")
plt.xlabel("Step")
plt.ylabel("Flocks")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cont=model.ras_control
for i in range(len(cont)):
    plt.figure(figsize=(12, 6))
    plt.imshow(cont[i], cmap="YlOrRd_r", vmin=0, vmax=10)
    plt.colorbar()
    plt.title("Rasters for year {}".format(i))
    plt.show()


In [ ]:
ret=model.return_raster
plt.figure(figsize=(12, 6))
plt.imshow(model.suitability_raster, cmap="YlGn", origin="upper", vmin=0, vmax=2, label="Suitability Raster")
plt.imshow(ret, cmap="gist_stern_r")
plt.colorbar()
plt.title("Activity Raster")
plt.show()

In [ ]:
# Visualizing the location of the first enclosure in the model
# model.enclosures[0] contains [(x, y), year, agent_object]
enc_pos =[]
for e in model.enclosures:
    enc_pos.append(e[0])


plt.figure(figsize=(10, 8))
plt.imshow(model.suitability_raster, cmap="YlGn", origin="upper", label="Suitability Raster")
for p in enc_pos:
    mesa_x, mesa_y = p
    # Convert from Mesa y (Cartesian) to numpy y
    numpy_y = to_numpy_y(model,mesa_y)
    plt.scatter(mesa_x, numpy_y, color="red", s=100)
plt.title(f"enclosures")
#plt.colorbar(label='Suitability')

plt.show()


# Calibration

## funcs

In [ ]:
def to_gdf(model_output,lower_left_x=139554.9251999901,lower_left_y=478515.53229999694,cell_size=250, plot=False):
    yo_sum=sum(model_output.yearly_outputs)
    yo_sum[yo_sum < 1] = 0
    yo_sum[(yo_sum > 1.5)]= 1
    # Alternative approach if model.enclosures contains the enclosure positions directly
    for enclosure in model_output.enclosures:
        # Get the enclosure position in Mesa coordinates
        mesa_x, mesa_y = enclosure[0]

        # Convert from Mesa y (Cartesian) to numpy y
        numpy_y = to_numpy_y(model_output,mesa_y)

        # Set the value to 2 for enclosure locations
        yo_sum[numpy_y, mesa_x] = 2
    #yo_sum[yo_sum > 2] = 2
    # Step 1: Find indices of cells with values > 0
    indices = np.column_stack((np.argwhere(yo_sum > 0), yo_sum[yo_sum > 0]))  # Shape: (N, 2), where N is the number of cells with value > 0

    # Step 2: Convert array indices to real-world coordinates
    real_world_coords = []
    for (row, col, value) in indices:
        # Calculate real-world coordinates
        x = lower_left_x + col * cell_size
        y = lower_left_y + (yo_sum.shape[0] - 1 - row) * cell_size
        real_world_coords.append((x, y, value))

    # Convert to a numpy array for easier manipulation
    real_world_coords = np.array(real_world_coords)
    # Step 3: Create a GeoDataFrame
    # Convert coordinates to Shapely Point objects
    geometry = [Point(x, y) for x, y, _ in real_world_coords]

    # Create a GeoDataFrame with the geometry and values
    gdf = gpd.GeoDataFrame({
        'geometry': geometry,
        'value': real_world_coords[:, 2]  # Include the values
    }, crs="EPSG:2039")
    if plot:
        # Plot using GeoPandas
        gdf.plot(column='value', cmap='viridis', legend=True, markersize=40, figsize=(7, 7))
        plt.title("Land Use in the Study Area")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.show()
    return gdf

In [ ]:
gdf_real=gpd.read_file(r'D:\itay\ABM\points_all\P_for_calib.shp')
def obj_func(gdf, gdf1, run_dir):
    def calculate_ellipse(points):
        mean_center = np.mean(points, axis=0)
        med_center = np.median(points, axis=0)

        # Calculate covariance matrix
        cov = np.cov(points, rowvar=False)

        # Calculate eigenvalues and eigenvectors
        eigenvalues, eigenvectors = linalg.eigh(cov)

        # Sort by eigenvalues in decreasing order
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Calculate standard deviations along axes (for 95% confidence ellipse)
        std_dev = np.sqrt(eigenvalues) * 2

        # Calculate rotation angle
        rotation = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])

        return mean_center, med_center, std_dev[0], std_dev[1], rotation

    # Assuming gdf and gdf1 are your GeoDataFrames
    # First, extract coordinates
    gdf['x'] = gdf['geometry'].x
    gdf['y'] = gdf['geometry'].y
    gdf1['x'] = gdf1['geometry'].x
    gdf1['y'] = gdf1['geometry'].y

    # Get points
    points = gdf[["x", "y"]].values
    points1 = gdf1[["x", "y"]].values

    # Calculate ellipse properties
    mean_center, med_center, major, minor, rotation = calculate_ellipse(points)
    mean_center1, med_center1, major1, minor1, rotation1 = calculate_ellipse(points1)

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot points from first GeoDataFrame - categorized by value
    gdf_value1 = gdf[gdf['value'] == 1]
    gdf_value2 = gdf[gdf['value'] == 2]
    ax.scatter(gdf_value1['x'], gdf_value1['y'], color='lightcoral', alpha=0.7, label='archaeology Value 1')
    ax.scatter(gdf_value2['x'], gdf_value2['y'], color='red', alpha=0.7, label='archaeology Value 2')

    # Plot points from second GeoDataFrame - categorized by value
    gdf1_value1 = gdf1[gdf1['value'] == 1]
    gdf1_value2 = gdf1[gdf1['value'] == 2]
    ax.scatter(gdf1_value1['x'], gdf1_value1['y'], color='lightblue', alpha=0.7, label='simulated Value 1')
    ax.scatter(gdf1_value2['x'], gdf1_value2['y'], color='blue', alpha=0.7, label='simulated Value 2')

    # Create and add ellipses to the plot
    ellipse = Ellipse(
        xy=mean_center,
        width=major * 2,
        height=minor * 2,
        angle=np.rad2deg(rotation),
        facecolor="none",
        edgecolor="red",
        linestyle="--",
        linewidth=2,
        label="Std. Ellipse archaeology"
    )
    ax.add_patch(ellipse)

    ellipse_1 = Ellipse(
        xy=mean_center1,
        width=major1 * 2,
        height=minor1 * 2,
        angle=np.rad2deg(rotation1),
        facecolor="none",
        edgecolor="blue",
        linestyle="--",
        linewidth=2,
        label="Std. Ellipse simulated"
    )
    ax.add_patch(ellipse_1)

    # Add mean centers to the plot
    ax.scatter(mean_center[0], mean_center[1], color='darkred', marker='*', s=100, label='archaeology Mean Center')
    ax.scatter(mean_center1[0], mean_center1[1], color='darkblue', marker='*', s=100, label='simulated Mean Center')

    # Set equal aspect so the ellipses look right
    ax.set_aspect('equal')

    # Add title and legend
    ax.set_title('Spatial Distribution Comparison with Standard Deviational Ellipses')
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

    # Calculate some metrics for the plot title or annotation
    vertices = ellipse.get_verts()
    ellipse_gdf = Polygon(vertices)
    vertices1 = ellipse_1.get_verts()
    ellipse_gdf1 = Polygon(vertices1)
    overlapping_area = ellipse_gdf.intersection(ellipse_gdf1).area
    total_non_overlapping_area = ellipse_gdf.symmetric_difference(ellipse_gdf1).area

    # Calculate overlap metric
    overlap_metric = (abs(overlapping_area - total_non_overlapping_area)) / (overlapping_area + total_non_overlapping_area) if (overlapping_area + total_non_overlapping_area) != 0 else 0
    overlap_score = 1 - overlap_metric

    # Add annotation with overlap information
    ax.annotate(f'Overlap score: {overlap_score:.4f}',
                xy=(0.5, 0.02),
                xycoords='axes fraction',
                ha='center',
                fontsize=12,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

    # Adjust layout
    plt.tight_layout()

    # Save the figure
    fig.savefig(os.path.join(run_dir, "spatial_similarity.png"))

    # Print additional information
    #print(f"archaeology Ellipse - Major axis: {major:.4f}, Minor axis: {minor:.4f}, Rotation: {np.rad2deg(rotation):.2f}°")
    #print(f"simulated Ellipse - Major axis: {major1:.4f}, Minor axis: {minor1:.4f}, Rotation: {np.rad2deg(rotation1):.2f}°")
    #print(f"Overlap area: {overlapping_area:.4f}")
    #print(f"Non-overlapping area: {total_non_overlapping_area:.4f}")
    #print(f"Spatial overlap score: {overlap_score:.4f}")

    # Calculate value ratio metrics
    t1 = gdf[gdf['value'] == 1].shape[0]
    t2 = gdf[gdf['value'] == 2].shape[0]
    t_ratio = t2 / t1 if t1 != 0 else 0

    t11 = gdf1[gdf1['value'] == 1].shape[0]
    t21 = gdf1[gdf1['value'] == 2].shape[0]
    t1_ratio = t21 / t11 if t11 != 0 else 0

    # Calculate the absolute difference between ratios
    diff = abs(t_ratio - t1_ratio)
    # Normalize to 0-1 scale where 1 means identical ratios
    ratio_enc = 1 - diff


    #print(f"archaeology Value ratio (2:1): {t_ratio:.4f}")
    #print(f"simulated Value ratio (2:1): {t1_ratio:.4f}")
    #print(f"Value distribution similarity: {ratio_enc:.4f}")

    # Calculate total score
    total = (overlap_score + ratio_enc) / 2
    plt.close(fig)
    #print(f"Total similarity score: {total:.4f}")
    return total, ratio_enc

mgdf=to_gdf(model)
rund=r'D:\itay\ABM\Results\runs_simp'
obj_func(gdf_real,mgdf,rund)

In [ ]:
def run_model_opt(model_years, ras_wts, main_run_directory, permanent_weights_dict=None, yearly_weights_dict=None, y_output=y_output, seed=None):
    """
    Runs the optimization version of the model.

    Args:
        model_years (int): Number of years to simulate.
        ras_wts (list): List of weights for environmental factors.
        main_run_directory (str): Base directory to store run results.
        permanent_weights_dict (dict, optional): Dictionary of permanent weights. Defaults to None.
        yearly_weights_dict (dict, optional): Dictionary of yearly weights. Defaults to None.
        y_output (list, optional): Yearly output data. Defaults to y_output.
        seed (int, optional): Random seed for reproducibility. Defaults to None.

    Returns:
        tuple: A tuple containing the completed model instance and the path to the run directory.
    """
    # Turn off interactive plotting globally at the beginning
    plt.ioff()

    # Generate a timestamp frthr the run directory
    current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    # Set seeds to control varying randomization
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # Create a list of indices and shuffle them for random yearly data selection
    inds = list(range(len(y_output)))
    random.shuffle(inds)

    # Generate suitability, return, and stress rasters for the initial year
    suitability_raster, return_raster, stress_ras = get_suitability_raster(y_output, inds, year=0, yearly_outputs=[], weights=ras_wts)

    # Initialize the model with the generated rasters
    model = NomadModel(suitability_raster, return_raster, stress_ras, inds,ras_w=ras_wts)

    # Create a directory for storing run results
    run_directory =os.path.join(main_run_directory,"run_{}".format(current_date))
    if not os.path.exists(run_directory):
        os.makedirs(run_directory)
    plot_y=[10,20,30,40,50,60,70]
    # Run the simulation for the specified number of years with progress bar
    for i in range(model_years):
        model.step()  # Advance the model by one year
        if i != (model_years - 1):
            if i in plot_y:
                viz_map(model, model.suitability_raster, i, run_directory)
            model.move_year(inds)
        else:
            viz_map(model, model.suitability_raster, i, run_directory)

    # Retrieve collected data from the model
    household_data = model.datacollector.get_agent_vars_dataframe()

    # Save data to CSV files
    household_data.to_csv(os.path.join(run_directory, "household_data.csv"))

    # Save weights to JSON files
    if permanent_weights_dict:
        with open(os.path.join(run_directory, "permanent_weights_dict.json"), "w") as f:
            json.dump(permanent_weights_dict, f)
    if yearly_weights_dict:
        with open(os.path.join(run_directory, "yearly_weights_dict.json"), "w") as f:
            json.dump(yearly_weights_dict, f)

    # Turn interactive mode back on at the end if needed for other visualizations
    #plt.ion()

    return model,run_directory

In [ ]:
def objective_function(permanent_weights, yearly_weights,trial_n, n_iterations=10,n_years=75, random_seed=None):
    """
    Your stochastic model objective function that returns a score from 0 to 1, where 1 is best.
    This function should use the weights provided in the dictionaries to compute the score.

    Args:
        permanent_weights_dict: Dictionary of permanent weights
        yearly_weights_dict: Dictionary of yearly weights
        n_iterations: Number of times to run the stochastic model
        random_seed: Optional random seed for reproducibility

    Returns:
        float: Average score across multiple runs (value between 0 and 1)
    """

    # Replace this with your actual implementation
    scores = []
    main_run_directory=r"D:\itay\ABM\Results\opt\trial_{}".format(trial_n)
    # Set random seed if provided
    if random_seed is not None:
        np.random.seed(random_seed)

    # Run the model multiple times to account for stochasticity
    for i in range(n_iterations):
        pre_w = {**permanent_weights, **yearly_weights}
        w=list(pre_w.values())
        if sum(w) != 1:
            #w/=sum(w)
            w = np.array(w) / sum(w)
        sim,run_dir=run_model_opt(n_years, w, main_run_directory, permanent_weights_dict=permanent_weights, yearly_weights_dict=yearly_weights)

        gdf_model=to_gdf(sim)
        gdf_real=gpd.read_file(r'D:\itay\ABM\points_all\P_for_calib.shp')
        score,ratio_enc=obj_func(gdf_real,gdf_model,run_dir)
        score=1-score
        scores.append(score)

    # Return the average score across all iterations
    return np.mean(scores)  # Returns value between 0 and 1


In [ ]:
def objective(trial,n_iter=10):
    """
    Optuna objective function that samples parameter values and evaluates the stochastic model.
    """
    # Sample parameters for permanent weights
    sum_w=0
    dist_to_kb = trial.suggest_int("dist_to_kb", 0, 7)
    sum_w+=dist_to_kb
    p_water = trial.suggest_int("p_water", 0,7)
    sum_w+=p_water
    mean_rain = trial.suggest_int("Mean_rain", 0,7)
    sum_w+=mean_rain
    slope_suitability = trial.suggest_int("slope_suitability", 0,7)
    sum_w+=slope_suitability
    return_to_site = trial.suggest_int("return_to_site", 0,7)
    sum_w+=return_to_site
    humen_stress = trial.suggest_int("humen_stress", 0,7)
    sum_w+=humen_stress
    yearly_rain = trial.suggest_int("Yearly_rain", 0,7)
    sum_w+=yearly_rain
    #p_agri = trial.suggest_int("p_agri", 0,7)
    #aspect = trial.suggest_int("aspect", 0,7)
    #p_veg_fit = trial.suggest_int("p_veg_fit", 0,7)
    #slope_range = trial.suggest_int("slope_range", 0,7)
    # Sample parameters for yearly weights
    #pasture_y = trial.suggest_int("pasture_y", 0,7)
    #yearly_agri = trial.suggest_int("yearly_agri", 0,7)
    #yearly_water = trial.suggest_int("yearly_water", 0,7)
    #normalize to 0-1
    dist_to_kb/=sum_w
    p_water/=sum_w
    mean_rain/=sum_w
    slope_suitability/=sum_w
    return_to_site/=sum_w
    humen_stress/=sum_w
    yearly_rain/=sum_w
    # Create dictionaries to store weights for permanent and yearly factors
    permanent_weights_dict = {
        "dist_to_kb": dist_to_kb,  # Distance to Kadesh Barnea
        "p_water": p_water,  # Permanent water sources
        "Mean_rain": mean_rain,  # Mean annual rainfall
        "slope_suitability": slope_suitability,  # Slope suitability
        #"p_agri": wp1,  # Agricultural suitability
        #"aspect": wp3,  # Terrain aspect
        #"p_veg_fit": wp5,  # Vegetation suitability
        #"slope_range": wp7,  # Slope range
    }
    yearly_weights_dict = {
        "return_to_site": return_to_site,  # Return to previous sites
        "humen_stress": humen_stress,  # Human stress
        "Yearly_rain": yearly_rain,  # Yearly rainfall
        #"pasture_y": ws3,  # Pasture yield
        #"yearly_agri": ws4,  # Yearly agricultural suitability
        #"yearly_water": ws5,  # Yearly water availability

    }

    print(permanent_weights_dict,yearly_weights_dict)

    # Evaluate the objective function with these parameters
    # Run multiple iterations to account for stochasticity
    # Fixed random seed for each trial ensures reproducibility within a trial
    # but allows for different results between trials
    score = objective_function(
        permanent_weights_dict,
        yearly_weights_dict,
        trial.number,
        n_years=75,
        n_iterations=n_iter,  # Run 10 iterations for each parameter set
        random_seed=trial.number  # Use trial number as seed for reproducibility
    )

    return score  # Optuna maximizes by default


In [ ]:
def run_optimization(n_trials=200, n_jobs=1, show_progress_bar=True):
    """
    Run the Optuna optimization process for a stochastic model.

    Args:
        n_trials: Number of optimization trials to run
        n_jobs: Number of parallel jobs to run (-1 uses all cores)
        show_progress_bar: Whether to show a progress bar

    Returns:
        The best parameters and the best score
    """
    local_storage = r"sqlite:///D:\itay\ABM\Results\opt\opt_study.db"

    # Create a pruner to terminate unpromising trials early
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    # Create a study object and optimize the objective function
    study = optuna.create_study(
        direction="minimize",
        storage=local_storage,
        study_name='trial2',
        load_if_exists=True,
        pruner=pruner,
        sampler=optuna.samplers.TPESampler(seed=42)  # Use TPE sampler with fixed seed for reproducibility
    )

    # Run optimization
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=show_progress_bar
    )

    # Get the best parameters
    best_params = study.best_params
    best_score = study.best_value

    # Extract the best weights into the dictionaries format
    best_permanent_weights = {
        #"p_agri": best_params["p_agri"],
        "dist_to_kb": best_params["dist_to_kb"],
        #"aspect": best_params["aspect"],
        "p_water": best_params["p_water"],
        #"p_veg_fit": best_params["p_veg_fit"],
        "Mean_rain": best_params["Mean_rain"],
        #"slope_range": best_params["slope_range"],
        "slope_suitability": best_params["slope_suitability"],
    }

    best_yearly_weights = {
        "return_to_site": best_params["return_to_site"],
        "humen_stress": best_params["humen_stress"],
        #"pasture_y": best_params["pasture_y"],
        #"yearly_agri": best_params["yearly_agri"],
        #"yearly_water": best_params["yearly_water"],
        "Yearly_rain": best_params["Yearly_rain"],
    }

    return best_permanent_weights, best_yearly_weights, best_score,study



## Run optimization

In [ ]:
best_permanent_weights, best_yearly_weights, best_score,study=run_optimization(n_trials=3, n_jobs=1, show_progress_bar=True)